# Projeto 01 - Xadrez

### Importando bibliotecas

In [1]:
import glfw
from OpenGL.GL import *
import OpenGL.GL.shaders
import numpy as np
import glm

### Inicializando a cena

In [2]:
# Cria a janela
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.TRUE);
window = glfw.create_window(800, 800, "Xadrez", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)

# Shaders em 3D
vertex_code = """
        attribute vec3 position;
        uniform mat4 mat_transformation;
        void main(){
            gl_Position = mat_transformation * vec4(position,1.0);
        }
        """

fragment_code = """
        uniform vec4 color;
        void main(){
            gl_FragColor = color;
        }
        """

# Aloca
program  = glCreateProgram()
vertex   = glCreateShader(GL_VERTEX_SHADER)
fragment = glCreateShader(GL_FRAGMENT_SHADER)

# Associa
glShaderSource(vertex, vertex_code)
glShaderSource(fragment, fragment_code)

# Compila
glCompileShader(vertex)
if not glGetShaderiv(vertex, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(vertex).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Vertex Shader")

glCompileShader(fragment)
if not glGetShaderiv(fragment, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(fragment).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Fragment Shader")

# Linkagem
glAttachShader(program, vertex)
glAttachShader(program, fragment)

glLinkProgram(program)
if not glGetProgramiv(program, GL_LINK_STATUS):
    print(glGetProgramInfoLog(program))
    raise RuntimeError('Linking error')

glUseProgram(program)

loc = glGetAttribLocation(program, "position")
loc_color = glGetUniformLocation(program, "color")
loc_transformation = glGetUniformLocation(program, "mat_transformation")

### Objetos

In [3]:
vertices = []

# Formas Geométricas

def cubo(c, lx, ly, lz):
    global vertices

    lx /= 2
    ly /= 2
    lz /= 2
    
    vertices += [
        # Bottom
        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]-ly, c[2]+lz),

        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]-ly, c[2]+lz),
        (c[0]-lx, c[1]-ly, c[2]+lz),

        # Top
        (c[0]-lx, c[1]+ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]+lz),
        (c[0]+lx, c[1]+ly, c[2]-lz),

        (c[0]-lx, c[1]+ly, c[2]-lz),
        (c[0]-lx, c[1]+ly, c[2]+lz),
        (c[0]+lx, c[1]+ly, c[2]+lz),

        # Front
        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]-lz),
        (c[0]+lx, c[1]-ly, c[2]-lz),

        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]-lx, c[1]+ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]-lz),

        # Back
        (c[0]-lx, c[1]-ly, c[2]+lz),
        (c[0]+lx, c[1]-ly, c[2]+lz),
        (c[0]+lx, c[1]+ly, c[2]+lz),

        (c[0]-lx, c[1]-ly, c[2]+lz),
        (c[0]+lx, c[1]+ly, c[2]+lz),
        (c[0]-lx, c[1]+ly, c[2]+lz),

        # Left
        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]-lx, c[1]-ly, c[2]+lz),
        (c[0]-lx, c[1]+ly, c[2]+lz),

        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]-lx, c[1]+ly, c[2]+lz),
        (c[0]-lx, c[1]+ly, c[2]-lz),

        # Right
        (c[0]+lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]+lz),
        (c[0]+lx, c[1]-ly, c[2]+lz),

        (c[0]+lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]+lz)
    ]

def prismaTri(c, lx, ly, lz):
    global vertices

    lx /= 2
    ly /= 2
    lz /= 2
    
    vertices += [
        # Bottom Tri
        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]-ly, c[2]-lz),
        (c[0]-lx, c[1]-ly, c[2]+lz),

        # Top Tri
        (c[0]-lx, c[1]+ly, c[2]-lz),
        (c[0]-lx, c[1]+ly, c[2]+lz),
        (c[0]+lx, c[1]+ly, c[2]-lz),

        # Back Rect
        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]-lz),
        (c[0]+lx, c[1]-ly, c[2]-lz),

        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]-lx, c[1]+ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]-lz),

        # Left Rect
        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]-lx, c[1]-ly, c[2]+lz),
        (c[0]-lx, c[1]+ly, c[2]+lz),

        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]-lx, c[1]+ly, c[2]+lz),
        (c[0]-lx, c[1]+ly, c[2]-lz),

        # Front Rect
        (c[0]-lx, c[1]-ly, c[2]+lz),
        (c[0]+lx, c[1]-ly, c[2]-lz),
        (c[0]-lx, c[1]+ly, c[2]+lz),

        (c[0]+lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]-lz),
        (c[0]-lx, c[1]+ly, c[2]+lz)
    ]

def trapz(c, l_m, l_M, h):
    global vertices

    l_m /= 2
    l_M /= 2
    h /= 2

    vertices += [
        # Bottom
        (c[0]-l_m, c[1]-h, c[2]-l_m),
        (c[0]+l_m, c[1]-h, c[2]-l_m),
        (c[0]+l_m, c[1]-h, c[2]+l_m),

        (c[0]-l_m, c[1]-h, c[2]-l_m),
        (c[0]+l_m, c[1]-h, c[2]+l_m),
        (c[0]-l_m, c[1]-h, c[2]+l_m),

        # Top
        (c[0]-l_M, c[1]+h, c[2]-l_M),
        (c[0]-l_M, c[1]+h, c[2]+l_M),
        (c[0]+l_M, c[1]+h, c[2]+l_M),

        (c[0]-l_M, c[1]+h, c[2]-l_M),
        (c[0]+l_M, c[1]+h, c[2]+l_M),
        (c[0]+l_M, c[1]+h, c[2]-l_M),

        # Front
        (c[0]-l_m, c[1]-h, c[2]-l_m),
        (c[0]-l_M, c[1]+h, c[2]-l_M),
        (c[0]+l_m, c[1]-h, c[2]-l_m),

        (c[0]+l_m, c[1]-h, c[2]-l_m),
        (c[0]-l_M, c[1]+h, c[2]-l_M),
        (c[0]+l_M, c[1]+h, c[2]-l_M),

        # Back
        (c[0]-l_m, c[1]-h, c[2]+l_m),
        (c[0]+l_m, c[1]-h, c[2]+l_m),
        (c[0]-l_M, c[1]+h, c[2]+l_M),

        (c[0]+l_m, c[1]-h, c[2]+l_m),
        (c[0]+l_M, c[1]+h, c[2]+l_M),
        (c[0]-l_M, c[1]+h, c[2]+l_M),

        # Left
        (c[0]-l_m, c[1]-h, c[2]-l_m),
        (c[0]-l_m, c[1]-h, c[2]+l_m),
        (c[0]-l_M, c[1]+h, c[2]+l_M),

        (c[0]-l_m, c[1]-h, c[2]-l_m),
        (c[0]-l_M, c[1]+h, c[2]+l_M),
        (c[0]-l_M, c[1]+h, c[2]-l_M),

        # Right
        (c[0]+l_m, c[1]-h, c[2]-l_m),
        (c[0]+l_M, c[1]+h, c[2]-l_M),
        (c[0]+l_M, c[1]+h, c[2]+l_M),

        (c[0]+l_m, c[1]-h, c[2]-l_m),
        (c[0]+l_M, c[1]+h, c[2]+l_M),
        (c[0]+l_m, c[1]-h, c[2]+l_m)
    ]

class Objeto:
    def __init__(self, id, inicio, quant, cor=[0,0,0], pos=[0,0]):
        self.id = id
        self.inicio = inicio
        self.quant = quant
        self.cor = cor
        self.pos = pos

objetos = []

In [4]:
COR = {
    "fundo": [0.784, 0.949, 0.933],
    "tabuleiro": {
        "base": [0.588, 0.392, 0.212],
        "brancas": [0.868, 0.879, 0.77],
        "pretas": [0.463, 0.584, 0.341]
    },
    "peça": {
        0: [1, 1, 1],
        1: [0.1, 0.1, 0.1]
    }
}

# Objetos (0 - Brancas / 1 - Pretas)
def add_peao(vertices, objetos, side, pos):
    inicio = len(vertices)
    cubo(grid[pos]+[0,0.0425,0], ds/2, 0.005, ds/2)
    cubo(grid[pos]+[0,0.054,0], ds/4, 0.018, ds/4)
    cubo(grid[pos]+[0,0.063+ds/7,0], ds/3.5, ds/3.5, ds/3.5)
    quant = len(vertices)-inicio
    id = "" if pos[0]==3 and pos[1]==1 else ("_branco" if side==0 else "_preto")
    objetos.append(Objeto("peao"+id, inicio, quant, COR["peça"][side], pos))

def add_cavalo(vertices, objetos, side, pos):
    dir = -1 if side==0 else 1
    inicio = len(vertices)
    cubo(grid[pos]+[0,0.0425,0], ds/2, 0.005, ds/2)
    cubo(grid[pos]+[0,0.054,dir*ds/12], ds/3.5, 0.018, ds/3.5)
    cubo(grid[pos]+[0,0.073,0], ds/3.5, 0.02, ds/2.2)
    prismaTri(grid[pos]+[dir*ds/10,0.088,dir*ds/10], ds/10, 0.01, ds/10)
    prismaTri(grid[pos]+[-dir*ds/10,0.088,dir*ds/10], -ds/10, -0.01, ds/10)
    quant = len(vertices)-inicio
    objetos.append(Objeto("cavalo_"+("branco" if side==0 else "preto"), inicio, quant, COR["peça"][side], pos))

def add_torre(vertices, objetos, side, pos):
    inicio = len(vertices)
    cubo(grid[pos]+[0,0.0425,0], ds/2, 0.005, ds/2)
    cubo(grid[pos]+[0,0.054,0], ds/4, 0.018, ds/4)
    cubo(grid[pos]+[0,0.063+ds/7,0], ds/3.5, ds/3.5, ds/3.5)
    cubo(grid[pos]+[0,0.0665,0], ds/3+0.002, 0.007, ds/3+0.002)
    cubo(grid[pos]+[-ds/10.5,0.063+ds/3,-ds/10.5], ds/10.5, ds/10.5, ds/10.5)
    cubo(grid[pos]+[-ds/10.5,0.063+ds/3,ds/10.5], ds/10.5, ds/10.5, ds/10.5)
    cubo(grid[pos]+[ds/10.5,0.063+ds/3,-ds/10.5], ds/10.5, ds/10.5, ds/10.5)
    cubo(grid[pos]+[ds/10.5,0.063+ds/3,ds/10.5], ds/10.5, ds/10.5, ds/10.5)
    quant = len(vertices)-inicio
    objetos.append(Objeto("torre_"+("branca" if side==0 else "preta"), inicio, quant, COR["peça"][side], pos))

def add_bispo(vertices, objetos, side, pos):
    inicio = len(vertices)
    cubo(grid[pos]+[0,0.0425,0], ds/2, 0.005, ds/2)
    cubo(grid[pos]+[0,0.055,0], ds/4, 0.02, ds/4)
    cubo(grid[pos]+[0,0.0685,0], ds/3+0.004, 0.007, ds/3+0.004)
    prismaTri(grid[pos]+[0.002,0.065+ds/6,0], -ds/3, ds/3, -ds/3)
    prismaTri(grid[pos]+[-0.002,0.065+ds/6,0], ds/3, ds/3, ds/3)
    cubo(grid[pos]+[0,0.065+19*ds/48,0], ds/8, ds/8, ds/8)
    quant = len(vertices)-inicio
    objetos.append(Objeto("bispo_"+("branco" if side==0 else "preto"), inicio, quant, COR["peça"][side], pos))

def add_rei(vertices, objetos, side, pos):
    inicio = len(vertices)
    cubo(grid[pos]+[0,0.0425,0], ds/2, 0.005, ds/2)
    cubo(grid[pos]+[0,0.063,0], ds/4, 0.036, ds/4)
    trapz(grid[pos]+[0,0.086,0], ds/4, ds/2, 0.01)
    trapz(grid[pos]+[0,0.0935,0], ds/2, ds/8, 0.005)
    cubo(grid[pos]+[0,0.1035,0], ds/8, 0.015, ds/8)
    cubo(grid[pos]+[0,0.1035,0], 0.013, ds/8, ds/8)
    quant = len(vertices)-inicio
    objetos.append(Objeto("rei_"+("branco" if side==0 else "preto"), inicio, quant, COR["peça"][side], pos))

def add_rainha(vertices, objetos, side, pos):
    inicio = len(vertices)
    cubo(grid[pos]+[0,0.0425,0], ds/2, 0.005, ds/2)
    cubo(grid[pos]+[0,0.063,0], ds/4, 0.036, ds/4)
    trapz(grid[pos]+[0,0.086,0], ds/4, ds/2, 0.01)
    cubo(grid[pos]+[0,0.091+ds/16,0], ds/8, ds/8, ds/8)
    quant = len(vertices)-inicio
    objetos.append(Objeto("rainha_"+("branca" if side==0 else "preta"), inicio, quant, COR["peça"][side], pos))

# Tabuleiro
grid = np.zeros((8, 8, 3), dtype=np.float32)
ds = 0.4/8
for i in range(8):
    for j in range(8):
        x = -0.2 + ds/2 + i*ds
        z = -0.2 + ds/2 + j*ds
        grid[i,j] = [x, -0.15, z]

        inicio = len(vertices)
        cubo([x, -0.15, z], ds, 0.08, ds)
        quant = len(vertices)-inicio
        if (i + j) % 2 != 0:
            objetos.append(Objeto("tabuleiro", inicio, quant, COR["tabuleiro"]["brancas"]))
        else:
            objetos.append(Objeto("tabuleiro", inicio, quant, COR["tabuleiro"]["pretas"]))

inicio = len(vertices)
cubo([0,-0.16,0], 0.5, 0.08, 0.5)
quant = len(vertices)-inicio
objetos.append(Objeto("tabuleiro", inicio, quant, COR["tabuleiro"]["base"]))

# Peões
for i in range(8):
    add_peao(vertices, objetos, 0, (i,1))
    add_peao(vertices, objetos, 1, (i,6))

# Torres
for i in range(2):
    add_torre(vertices, objetos, 0, (7*i,0))
    add_torre(vertices, objetos, 1, (7*i,7))

# Cavalos
for i in range(2):
    add_cavalo(vertices, objetos, 0, (1+5*i,0))
    add_cavalo(vertices, objetos, 1, (1+5*i,7))

# Bispos
for i in range(2):
    add_bispo(vertices, objetos, 0, (2+3*i,0))
    add_bispo(vertices, objetos, 1, (2+3*i,7))

# Reis
add_rei(vertices, objetos, 0, (3,0))
add_rei(vertices, objetos, 1, (3,7))


# Rainhas
add_rainha(vertices, objetos, 0, (4,0))
add_rainha(vertices, objetos, 1, (4,7))


vertices = np.array(vertices, dtype=np.float32)

### Configuração do buffer

In [5]:
buffer_VBO = glGenBuffers(1)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)

glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_DYNAMIC_DRAW)

glEnableVertexAttribArray(loc)

stride = vertices.strides[0]
offset = ctypes.c_void_p(0)
glVertexAttribPointer(loc, 3, GL_FLOAT, False, stride, offset)

### Loop principal

In [6]:
dt = 0.05

# Variáveis para translação
t_x = t_z = 0

# Variável para rotação
a_x = a_y = 0

# Variável para escala uniforme
sc = 0


# Variáveis para incrementos relativos à translação
vt_x = vt_z = 0

# Variáveis para incrementos relativos à rotação
va_x = va_y = 0

# Variável para incrementos relativos à escala uniforme
vsc = 0

# Variável para visualização poligonal
poly = False

# COMANDOS:
#   WASD - Movimentação do peão branco
#   QE - Rotação horizontal do tabuleiro
#   RF - Rotação vertical do tabuleiro
#   ZX - Zoom do tabuleiro
#   P - Ativa/Desativa a visualização poligonal
#   Espaço - Reseta as condições iniciais
def key_event(window,key,scancode,action,mods):
    global t_x, t_z, vt_x, vt_z
    global a_x, a_y, va_x, va_y
    global sc, vsc
    global poly

    if action == glfw.PRESS or action == glfw.REPEAT:
        if key == glfw.KEY_W: vt_z = 0.1*dt
        if key == glfw.KEY_S: vt_z = -0.1*dt
        if key == glfw.KEY_A: vt_x = -0.1*dt
        if key == glfw.KEY_D: vt_x = 0.1*dt

        if key == glfw.KEY_R: va_x = dt
        if key == glfw.KEY_F: va_x = -dt
        
        if key == glfw.KEY_E: va_y = dt
        if key == glfw.KEY_Q: va_y = -dt
    
        if key == glfw.KEY_X: vsc = dt
        if key == glfw.KEY_Z: vsc = -dt

        if key == glfw.KEY_SPACE:
            t_x = t_z = 0
            a_x = a_y = 0
            sc = 0
            poly = False

        if key == glfw.KEY_P: poly = not poly

    elif action == glfw.RELEASE:
        if key in (glfw.KEY_A, glfw.KEY_D): vt_x = 0
        if key in (glfw.KEY_W, glfw.KEY_S): vt_z = 0
        if key in (glfw.KEY_R, glfw.KEY_F): va_x = 0
        if key in (glfw.KEY_Q, glfw.KEY_E): va_y = 0
        if key in (glfw.KEY_Z, glfw.KEY_X): vsc = 0


glfw.set_key_callback(window,key_event)

In [7]:
# Matrizes de transformação 3D
def multiplica_matriz(a,b):
    m_a = a.reshape(4,4)
    m_b = b.reshape(4,4)
    m_c = np.dot(m_a,m_b)
    c = m_c.reshape(1,16)
    return c

def translacao(tx, ty, tz):
    return np.array([
        1, 0, 0, tx,
        0, 1, 0, ty,
        0, 0, 1, tz,
        0, 0, 0,  1
    ], np.float32)

def rotacao(rx, ry, rz):
    c, s = np.cos(rx), np.sin(rx)
    X = np.array([
        1, 0,  0, 0,
        0, c, -s, 0,
        0, s,  c, 0,
        0, 0,  0, 1
    ], np.float32)

    c, s = np.cos(ry), np.sin(ry)
    Y = np.array([
         c, 0, s, 0,
         0, 1, 0, 0,
        -s, 0, c, 0,
         0, 0, 0, 1
    ], np.float32)

    c, s = np.cos(rz), np.sin(rz)
    Z = np.array([
        c, -s, 0, 0,
        s,  c, 0, 0,
        0,  0, 1, 0,
        0,  0, 0, 1
    ], np.float32)

    YX = multiplica_matriz(Y,X)
    return multiplica_matriz(Z,YX)

def escala(sx, sy, sz):
    return np.array([
        sx,  0,  0, 0,
         0, sy,  0, 0,
         0,  0, sz, 0,
         0,  0,  0, 1
    ], np.float32)

In [8]:
glfw.show_window(window)

glEnable(GL_DEPTH_TEST)

# Centroide
cx, cy, cz = vertices.mean(axis=0)

while not glfw.window_should_close(window):
    glPolygonMode(GL_FRONT_AND_BACK,GL_LINE if poly else GL_FILL)
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    glClearColor(COR["fundo"][0], COR["fundo"][1], COR["fundo"][2], 1.0)
    

    t_x += vt_x; t_z += vt_z
    a_y += va_y
    a_x += va_x
    if ((2+sc > 0.7 and 2+sc < 3.5) or sc*vsc <= 0): sc += vsc # Limita o zoom

    # Limita a movimentação do peão dentro do tabuleiro
    t_x = np.clip(grid[3,1][0]+t_x, -0.175, 0.175)-grid[3,1][0]
    t_z = np.clip(grid[3,1][2]+t_z, -0.175, 0.175)-grid[3,1][2]

    for obj in objetos:
        # Movimenta um peão
        if obj.id == "peao": mat_transformation = translacao(t_x,0,t_z)
        else: mat_transformation = np.identity(4, np.float32)

        # Faz um zoom na peça que o peão está em cima
        if obj.id not in ["tabuleiro", "peao"]:
            pos_peao = ((grid[3,1][0]+t_x+0.2125)//ds, (grid[3,1][1]+t_z+0.2125)//ds)
            if pos_peao == obj.pos:
                px, py, pz = grid[obj.pos]
                mat_transformation = multiplica_matriz(translacao(-px,-py,-pz), mat_transformation)
                mat_transformation = multiplica_matriz(escala(2,2,2), mat_transformation)
                mat_transformation = multiplica_matriz(translacao(px,py,pz), mat_transformation)

        # Escala e rotação do tabuleiro
        mat_transformation = multiplica_matriz(translacao(-cx,-cy,-cz), mat_transformation)
        mat_transformation = multiplica_matriz(escala(2+sc,2+sc,2+sc), mat_transformation)
        mat_transformation = multiplica_matriz(rotacao(-np.pi/8+a_x,a_y,0), mat_transformation)
        mat_transformation = multiplica_matriz(translacao(cx,cy,cz), mat_transformation)

        
        glUniformMatrix4fv(loc_transformation, 1, GL_TRUE, mat_transformation) 
        glUniform4f(loc_color, obj.cor[0], obj.cor[1], obj.cor[2], 1.0)
        glDrawArrays(GL_TRIANGLES, obj.inicio, obj.quant)

    
    glfw.swap_buffers(window)
    glfw.poll_events()

glfw.terminate()